In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/archive (3)"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/archive (3)'

In [ ]:
import librosa
import pandas as pd

DATASET_PATH = "/content/drive/MyDrive/archive (3)"

fake_folders = ["xTTS", "seedtts_files", "VoiceBox", "VALLE",
                "PromptTTS2", "OpenAI", "FlashSpeech", "NaturalSpeech3"]
data = []

# Scan real
real_path = os.path.join(DATASET_PATH, "real_samples")
for f in os.listdir(real_path):
    if f.endswith(".wav"):
        data.append({"file": os.path.join(real_path, f), "label": "real", "source": "real_samples"})

# Scan fake
for folder in fake_folders:
    folder_path = os.path.join(DATASET_PATH, folder)
    if os.path.exists(folder_path):
        for f in os.listdir(folder_path):
            if f.endswith(".wav"):
                data.append({"file": os.path.join(folder_path, f), "label": "fake", "source": folder})

df = pd.DataFrame(data)
print(df["label"].value_counts())
print("---")
print(df["source"].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/archive (3)/real_samples'

In [ ]:
masalah = []
data_bersih = []

for idx, row in df.iterrows():
    try:
        y, sr = librosa.load(row["file"], sr=None)
        durasi = len(y) / sr
        if durasi < 1.0:
            masalah.append({"file": row["file"], "masalah": "terlalu pendek"})
        else:
            data_bersih.append(row)  # file ini aman
    except Exception as e:
        masalah.append({"file": row["file"], "masalah": "file rusak"})

df_bersih = pd.DataFrame(data_bersih)
print(f"✅ Data bersih: {len(df_bersih)} file")
print(f"❌ File bermasalah: {len(masalah)} file")

NameError: name 'df' is not defined

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_PATH = "/content/drive/MyDrive/melspec_output"

# Buat folder output
for label in ["real", "fake"]:
    os.makedirs(os.path.join(OUTPUT_PATH, label), exist_ok=True)

def convert_to_melspec(file_path, save_path):
    y, sr = librosa.load(file_path, sr=22050, duration=3.0)
    # Padding kalau durasi < 3 detik
    target_len = 22050 * 3
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    plt.figure(figsize=(2.24, 2.24), dpi=100)
    librosa.display.specshow(mel_db, sr=sr, fmax=8000)
    plt.axis('off')
    plt.tight_layout(pad=0)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close()

# Konversi semua file
for idx, row in df_bersih.iterrows():
    label_folder = os.path.join(OUTPUT_PATH, row["label"])
    filename = f"{row['source']}_{os.path.basename(row['file']).replace('.wav', '.png')}"
    save_path = os.path.join(label_folder, filename)
    convert_to_melspec(row["file"], save_path)
    if idx % 100 == 0:
        print(f"Progress: {idx}/{len(df_bersih)} file diproses...")

print("✅ Semua selesai dikonversi!")

NameError: name 'df_bersih' is not defined

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

SPLIT_OUTPUT = "/content/drive/MyDrive/dataset_split"

# Buat struktur folder
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        os.makedirs(f"{SPLIT_OUTPUT}/{split}/{label}", exist_ok=True)

# Ambil list semua gambar
all_images = []
for label in ["real", "fake"]:
    folder = os.path.join(OUTPUT_PATH, label)
    for f in os.listdir(folder):
        if f.endswith(".png"):
            all_images.append({"file": os.path.join(folder, f), "label": label})

df_img = pd.DataFrame(all_images)

# Split 70% train, 15% val, 15% test
train, temp = train_test_split(df_img, test_size=0.3, stratify=df_img["label"], random_state=42)
val, test = train_test_split(temp, test_size=0.5, stratify=temp["label"], random_state=42)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

# Salin file ke folder masing-masing
for split_name, split_df in [("train", train), ("val", val), ("test", test)]:
    for _, row in split_df.iterrows():
        dst = os.path.join(SPLIT_OUTPUT, split_name, row["label"], os.path.basename(row["file"]))
        shutil.copy(row["file"], dst)

print("Split selesai.")

KeyError: 'label'

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATASET_PATH = "/content/drive/MyDrive/dataset_split"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    f"{DATASET_PATH}/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_data = datagen.flow_from_directory(
    f"{DATASET_PATH}/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_data = datagen.flow_from_directory(
    f"{DATASET_PATH}/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 0 images belonging to 2 classes.
Found 0 images belonging to 2 classes.
Found 0 images belonging to 2 classes.
